# 07 — Genre classifier (roadmap A10)

Routes each snippet to the right transcription model: **electronic → v2mix
(fine-tuned)**, **classic / acoustic band → base YourMT3+**. The classifier is
the **YourMT3+ encoder** (spectrogram → Perceiver-TF latents, pretrained,
frozen) **with the decoder replaced by a small classification head**.

**Classes** (labels come from which dataset a file belongs to — no manual
labeling):

| class | datasets | routed model |
|---|---|---|
| `electronic` | strudel (corpus + synthetic), NES-MDB | v2mix_s42 |
| `classic` | MAESTRO | base |
| `acoustic_band` | Slakh | base |
| `electronic_drums` | EGMD | v2mix_s42 |

**Ground rules of this notebook**
- Uses **exactly the fine-tuning train/validation/test splits** (the
  `yourmt3_indexes/*_file_list.json` files nb05 trained from) — no new split
  logic, so the strudel split stays repo-level leak-free.
- Training input = random **5-second crops**, 16 kHz mono — the same audio
  format the app's GPU worker receives.
- Everything runs **in the cell you're looking at** (no background drivers);
  every step prints its own validation. Helpers live in one cell so the main
  flow stays short.

Runtime: any Colab GPU (A100/L4/T4 all fine — the encoder is frozen).


## 1. Setup — Drive, model code, imports


In [ ]:
# Environment: Colab (Drive-mounted) or a local checkout with the datasets.
import json, math, os, random, sys, time
from collections import Counter, defaultdict
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/restrudel")
    %pip -q install "transformers==4.39.3" einops mido librosa soundfile \
        "pytorch-lightning>=2.2.1" wandb deprecated smart-open \
        "git+https://github.com/craffel/mir_eval.git" \
        "git+https://github.com/katsura-jp/pytorch-cosine-annealing-with-warmup.git"
else:
    DRIVE = Path(os.environ.get("RESTRUDEL_DRIVE",
                                Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

DATA_HOME = DRIVE / "datasets"
INDEX_DIR = DATA_HOME / "yourmt3_indexes"
MODEL_ROOT = (Path("/content") if IN_COLAB else DRIVE) / "models" / "YourMT3"
DRIVE_MODEL_CACHE = DRIVE / "models" / "YourMT3"      # persistent base-model cache
OUT_DIR = DRIVE / "models" / "classifier"             # exported classifier goes here

import numpy as np
import torch
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"colab={IN_COLAB}  device={DEVICE}")
print(f"data_home={DATA_HOME}")
assert INDEX_DIR.exists(), "index dir missing — the fine-tuning datasets must be in Drive (notebook 04)"


In [ ]:
# Fetch the YourMT3 code + base checkpoint (Drive-cached; HF one-time fallback),
# then make the source importable. `model_helper.py` is the loader the GPU
# worker uses in production — we load the model the exact same way.
import shutil

AMT_SRC = MODEL_ROOT / "amt" / "src"
RELEASED_EXP = "mc13_256_g4_all_v7_mt3f_sqr_rms_moe_wf4_n8k2_silu_rope_rp_b36_nops"
RELEASED_CKPT = MODEL_ROOT / "amt" / "logs" / "2024" / RELEASED_EXP / "checkpoints" / "last.ckpt"
ARCH = ["-tk", "mc13_full_plus_256", "-dec", "multi-t5", "-nl", "26",
        "-enc", "perceiver-tf", "-sqr", "1", "-ff", "moe", "-wf", "4",
        "-nmoe", "8", "-kmoe", "2", "-act", "silu", "-epe", "rope", "-rp", "1",
        "-ac", "spec", "-hop", "300", "-atc", "1"]

def ensure(rel, hf_patterns):
    """Ensure MODEL_ROOT/<rel> exists: Drive cache first, else HF snapshot (then cache)."""
    dst = MODEL_ROOT / rel
    if dst.exists():
        return print(f"  ready: {rel}")
    cached = DRIVE_MODEL_CACHE / rel
    if cached.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        (shutil.copytree if cached.is_dir() else shutil.copy2)(cached, dst)
        return print(f"  from Drive cache: {rel}")
    from huggingface_hub import snapshot_download
    repo_type = "model" if rel.startswith("amt/logs") else "space"
    snapshot_download(repo_id="mimbres/YourMT3", repo_type=repo_type,
                      allow_patterns=hf_patterns,
                      local_dir=str(MODEL_ROOT / ("amt" if repo_type == "model" else "")))
    cached.parent.mkdir(parents=True, exist_ok=True)
    (shutil.copytree if dst.is_dir() else shutil.copy2)(dst, cached)
    print(f"  from HF (now cached): {rel}")

ensure("amt/src", ["amt/src/**"])
ensure("model_helper.py", ["model_helper.py"])
ensure(f"amt/logs/2024/{RELEASED_EXP}/checkpoints/last.ckpt", [f"logs/2024/{RELEASED_EXP}/**"])

for p in (str(AMT_SRC), str(MODEL_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

# One tiny compat patch (idempotent): torch>=2.6 defaults torch.load to
# weights_only=True; the trusted Lightning checkpoint needs the old behaviour.
mh = MODEL_ROOT / "model_helper.py"
t = mh.read_text()
if "weights_only" not in t and "torch.load(" in t:
    mh.write_text(t.replace("torch.load(", "torch.load(weights_only=False, ", 1))
    print("  patched model_helper torch.load")

print("model code + checkpoint ready:", RELEASED_CKPT.exists())


## 2. Helpers

Everything reusable lives here so the flow below stays short. Read top to
bottom once — each group is commented.


In [ ]:
import soundfile as sf

# ---- classes & routing --------------------------------------------------------
CLASSES = ["electronic", "classic", "acoustic_band", "electronic_drums"]
DATASET_TO_CLASS = {"strudel": "electronic", "nesmdb": "electronic",
                    "maestro": "classic", "slakh": "acoustic_band",
                    "egmd": "electronic_drums"}
CLASS_TO_MODEL = {"electronic": "v2mix", "classic": "base",
                  "acoustic_band": "base", "electronic_drums": "v2mix"}

# ---- audio constants (match the app's worker input) ---------------------------
SR = 16_000
CROP_SEC = 5.0
CROP_SAMPLES = int(SR * CROP_SEC)
RMS_GATE = 3e-3          # crops quieter than this are re-rolled (silence has no class)

# ---- file lists: EXACTLY the fine-tuning splits -------------------------------
def rebase(p):
    """Index paths are absolute from the build machine — rebase onto DATA_HOME."""
    s = str(p)
    marker = "/datasets/"
    return DATA_HOME / s.split(marker, 1)[1] if marker in s else Path(s)

def load_split(split):
    """-> list[(wav_path, class_idx)] from the yourmt3 index files of `split`."""
    items = []
    for ds, cls in DATASET_TO_CLASS.items():
        fp = INDEX_DIR / f"{ds}_{split}_file_list.json"
        if not fp.exists():
            continue
        for e in json.load(open(fp)).values():
            wav = rebase(e["mix_audio_file"])
            items.append((wav, CLASSES.index(cls)))
    return items

# ---- robust 5 s crop loading (soundfile: version-stable random access) --------
def _retry(fn, tries=5):
    """Drive FUSE sporadically throws OSError on fine files — retry w/ backoff."""
    for i in range(tries):
        try:
            return fn()
        except (OSError, sf.LibsndfileError):
            if i == tries - 1:
                raise
            time.sleep(2 ** i)

def load_crop(path, deterministic_k=None, tries=5):
    """One 5 s mono 16 kHz crop as a float tensor. Random offset for training;
    pass deterministic_k=(i, n) for the i-th of n evenly spaced crops (val/test,
    reproducible). Re-rolls near-silent random crops, pads short files.
    All dataset WAVs are already 16 kHz mono (the yourmt3_16k format)."""
    info = _retry(lambda: sf.info(str(path)))
    total = info.frames
    assert info.samplerate == SR, f"{path}: {info.samplerate} Hz (expected {SR})"
    for _ in range(tries):
        if total <= CROP_SAMPLES:
            off = 0
        elif deterministic_k is not None:
            i, n = deterministic_k
            off = int((total - CROP_SAMPLES) * (i / max(1, n - 1)))
        else:
            off = random.randint(0, total - CROP_SAMPLES)
        data = _retry(lambda: sf.read(str(path), start=off, frames=CROP_SAMPLES,
                                      dtype="float32", always_2d=True)[0])
        wav = torch.from_numpy(data.mean(axis=1))   # -> mono
        if wav.numel() < CROP_SAMPLES:              # short file -> pad
            wav = torch.nn.functional.pad(wav, (0, CROP_SAMPLES - wav.numel()))
        if deterministic_k is not None or wav.pow(2).mean().sqrt() > RMS_GATE:
            return wav
    return wav                                      # all rolls quiet: accept the last

# ---- balanced epoch sampling --------------------------------------------------
def balanced_epoch(items, n_per_class, rng):
    """Sample n_per_class (path, label) pairs per class, with replacement."""
    by_class = defaultdict(list)
    for it in items:
        by_class[it[1]].append(it)
    picks = []
    for c in range(len(CLASSES)):
        picks += [rng.choice(by_class[c]) for _ in range(n_per_class)]
    rng.shuffle(picks)
    return picks

def balanced_subset(items, n_per_class, seed=0):
    """A FIXED balanced subset (no replacement, capped per class). Used for the
    per-epoch validation check and the test sweep: EGMD alone has thousands of
    files, and Drive random reads — not the GPU — dominate wall time. The same
    seed returns the same subset every call, so epoch numbers stay comparable."""
    by_class = defaultdict(list)
    for it in items:
        by_class[it[1]].append(it)
    r = random.Random(seed)
    picks = []
    for c in range(len(CLASSES)):
        pool = by_class[c][:]
        r.shuffle(pool)
        picks += pool[:n_per_class]
    return picks

# ---- encoder + head -----------------------------------------------------------
def encode_pooled(model, wavs):
    """(B, CROP_SAMPLES) waveforms -> (B, D) pooled encoder latents.

    The 5 s crop spans ~3 of the model's native ~2 s segments; each segment
    goes through spectrogram -> pre_encoder -> encoder, then everything but
    the batch and feature dims is mean-pooled."""
    B = wavs.shape[0]
    seg_len = model.audio_cfg["input_frames"]
    n_seg = math.ceil(CROP_SAMPLES / seg_len)
    padded = torch.nn.functional.pad(wavs, (0, n_seg * seg_len - wavs.shape[1]))
    segs = padded.view(B * n_seg, 1, seg_len)                    # (B*n, 1, T)
    x = model.spectrogram(segs)
    x = model.pre_encoder(x)
    enc = model.encoder(inputs_embeds=x)["last_hidden_state"]    # (B*n, ..., D)
    enc = enc.flatten(1, -2).mean(dim=1)                         # pool all mid dims
    return enc.view(B, n_seg, -1).mean(dim=1)                    # pool segments

class GenreHead(torch.nn.Module):
    """The replacement for YourMT3+'s decoder: pooled latents -> class logits.
    LazyLinear infers the latent dim D from the first batch — nothing hardcoded."""
    def __init__(self, n_classes=len(CLASSES)):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Dropout(0.1),
            torch.nn.LazyLinear(n_classes),
        )
    def forward(self, feats):
        feats = torch.nn.functional.layer_norm(feats, feats.shape[-1:])
        return self.net(feats)

# ---- evaluation printing ------------------------------------------------------
def confusion(y_true, y_pred):
    m = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
    for t_, p_ in zip(y_true, y_pred):
        m[t_][p_] += 1
    return m

def print_report(y_true, y_pred, title):
    m = confusion(y_true, y_pred)
    print(f"-- {title} --")
    header = " " * 18 + "".join(f"{c[:12]:>14}" for c in CLASSES)
    print(header + f"{'recall':>9}")
    f1s = []
    for i, c in enumerate(CLASSES):
        rec = m[i, i] / m[i].sum() if m[i].sum() else float("nan")
        prec = m[i, i] / m[:, i].sum() if m[:, i].sum() else float("nan")
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        f1s.append(f1)
        print(f"{c:<18}" + "".join(f"{v:>14}" for v in m[i]) + f"{rec:>9.3f}")
    acc = np.trace(m) / m.sum()
    print(f"accuracy {acc:.3f} · macro-F1 {np.nanmean(f1s):.3f}")
    return acc, float(np.nanmean(f1s))

print(f"{len(CLASSES)} classes: {CLASSES}")
print("crop:", CROP_SEC, "s =", CROP_SAMPLES, "samples @", SR, "Hz")


## 3. Data — the fine-tuning splits, per-class counts

Labels are just *which dataset a file comes from*. Nothing is re-split.


In [ ]:
train_items = load_split("train")
val_items   = load_split("validation")
test_items  = load_split("test")

for name, items in [("train", train_items), ("validation", val_items), ("test", test_items)]:
    cnt = Counter(CLASSES[c] for _, c in items)
    print(f"{name:<12}{sum(cnt.values()):>7} files   " +
          "  ".join(f"{c}={cnt.get(c, 0)}" for c in CLASSES))

# validation of the loader: decode one crop per class and show shape + RMS
rng = random.Random(SEED)
for cls_idx, cls in enumerate(CLASSES):
    path, _ = next(it for it in train_items if it[1] == cls_idx)
    w = load_crop(path)
    print(f"  sample [{cls}] {Path(path).parent.name}: shape={tuple(w.shape)}, "
          f"rms={w.pow(2).mean().sqrt():.4f}")


## 4. Model — frozen YourMT3+ encoder, new classification head

Loaded exactly the way the production GPU worker loads it
(`model_helper.load_model_checkpoint`), then the decoder is simply never
called — `encode_pooled` stops after the encoder.


In [ ]:
os.chdir(MODEL_ROOT)   # the loader resolves logs/ relative to cwd
from model_helper import load_model_checkpoint

t0 = time.time()
backbone = load_model_checkpoint(
    args=[f"{RELEASED_EXP}@last.ckpt", "-p", "2024", *ARCH,
          "-pr", "bf16-mixed" if DEVICE == "cuda" else "32"],
    device=DEVICE)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad_(False)                     # stage 1: linear probe
print(f"backbone loaded+frozen in {time.time()-t0:.0f}s")

head = GenreHead().to(DEVICE)

# validation: one dummy batch end-to-end, then parameter counts
with torch.no_grad():
    feats = encode_pooled(backbone, torch.zeros(2, CROP_SAMPLES, device=DEVICE))
    logits = head(feats)
n_frozen = sum(p.numel() for p in backbone.parameters())
n_train  = sum(p.numel() for p in head.parameters())
print(f"latent dim D={feats.shape[-1]} -> logits {tuple(logits.shape)}")
print(f"params: frozen backbone {n_frozen/1e6:.1f} M · trainable head {n_train}")


## 5. Stage 1 — train the head (encoder frozen)

Plain in-cell loop. Per epoch: fresh balanced sample of 5 s crops, one pass,
then the validation confusion matrix — printed right here. Best head (by val
macro-F1) is kept in memory and mirrored to Drive.


In [ ]:
N_PER_CLASS = 2000        # crops per class per epoch
BATCH = 32                # waveform crops per step (each becomes ~3 encoder segments)
EPOCHS = 10
PATIENCE = 3
VAL_PER_CLASS = 150       # fixed balanced val subset per epoch (Drive IO is the
                          # bottleneck, not the GPU — full val runs once in §7)

opt = torch.optim.AdamW(head.parameters(), lr=1e-3)
rng = random.Random(SEED)
val_epoch_items = balanced_subset(val_items, VAL_PER_CLASS, seed=SEED)
print(f"per-epoch val subset: {len(val_epoch_items)} files "
      f"({Counter(CLASSES[c] for _, c in val_epoch_items)})")
history, best = [], {"f1": -1.0, "state": None, "epoch": -1}

def run_eval(items, crops_per_file):
    head.eval()
    ys, ps = [], []
    with torch.no_grad():
        for i0 in range(0, len(items), BATCH):
            chunk = items[i0:i0 + BATCH]
            for k in range(crops_per_file):
                wavs = torch.stack([load_crop(p, deterministic_k=(k, crops_per_file))
                                    for p, _ in chunk]).to(DEVICE)
                logits = head(encode_pooled(backbone, wavs))
                ps += logits.argmax(-1).cpu().tolist()
                ys += [c for _, c in chunk]
    head.train()
    return ys, ps

for epoch in range(1, EPOCHS + 1):
    t0, losses = time.time(), []
    picks = balanced_epoch(train_items, N_PER_CLASS, rng)
    head.train()
    for i0 in range(0, len(picks), BATCH):
        chunk = picks[i0:i0 + BATCH]
        wavs = torch.stack([load_crop(p) for p, _ in chunk]).to(DEVICE)
        labels = torch.tensor([c for _, c in chunk], device=DEVICE)
        with torch.no_grad():
            feats = encode_pooled(backbone, wavs)
        loss = torch.nn.functional.cross_entropy(head(feats), labels)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())

    ys, ps = run_eval(val_epoch_items, 1)
    acc, f1 = print_report(ys, ps, f"epoch {epoch} · loss {np.mean(losses):.4f} "
                                   f"· {time.time()-t0:.0f}s")
    history.append({"epoch": epoch, "loss": float(np.mean(losses)),
                    "val_acc": acc, "val_macro_f1": f1})
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "training_history.json").write_text(json.dumps(history, indent=2))
    if f1 > best["f1"]:
        best = {"f1": f1, "state": {k: v.cpu().clone() for k, v in head.state_dict().items()},
                "epoch": epoch}
        torch.save(best["state"], OUT_DIR / "head_best.pt")
    elif epoch - best["epoch"] >= PATIENCE:
        print(f"early stop — no val improvement for {PATIENCE} epochs"); break

head.load_state_dict(best["state"])
plt.plot([h["epoch"] for h in history], [h["loss"] for h in history], label="train loss")
plt.plot([h["epoch"] for h in history], [h["val_macro_f1"] for h in history], label="val macro-F1")
plt.xlabel("epoch"); plt.legend(); plt.grid(alpha=0.3)
plt.title(f"stage 1 — best val macro-F1 {best['f1']:.3f} @ epoch {best['epoch']}")
plt.show()


## 6. Stage 2 (conditional) — partial unfreeze

**Run only if stage 1's val macro-F1 is below ~0.95.** Unfreezes the last two
encoder blocks at a tiny LR. Skipping this cell is the expected path.


In [ ]:
RUN_STAGE_2 = False   # flip manually if stage 1 missed the gate

if RUN_STAGE_2:
    blocks = list(backbone.encoder.children())[-2:]
    for b in blocks:
        for p in b.parameters():
            p.requires_grad_(True)
    opt = torch.optim.AdamW(
        [{"params": head.parameters(), "lr": 1e-4},
         {"params": [p for b in blocks for p in b.parameters()], "lr": 1e-5}])
    print("stage 2 enabled:",
          sum(p.numel() for b in blocks for p in b.parameters()) / 1e6, "M params unfrozen")
    # re-run the stage-1 loop cell after this one (it picks up the new `opt`);
    # encode_pooled must then run WITH grad: wrap the loop's no_grad accordingly.
else:
    print("stage 2 skipped (stage 1 met the gate)")


## 7. Test evaluation — accuracy, routing regret, decision bias τ

Run **once**, after training decisions are frozen. Two views:
1. the plain classifier report on the test splits;
2. what actually matters — **routed transcription quality**, using the B8
   per-category F1 of each model. Misrouting electronic→base is catastrophic
   (−0.3…−0.5 F1), classical→v2 is mild (−0.1), so we bias routing toward
   v2mix: route to base only when P(classic)+P(acoustic_band) > τ.


In [ ]:
TEST_CROPS_PER_FILE = 3
TEST_PER_CLASS = 500     # cap per class (EGMD alone has 5k+ test files; Drive IO
                         # dominates). Balanced + deterministic, so the report is
                         # comparable run to run; strudel/maestro/slakh fit fully.

test_eval_items = balanced_subset(test_items, TEST_PER_CLASS, seed=SEED)
print(f"test subset: {len(test_eval_items)} files "
      f"({Counter(CLASSES[c] for _, c in test_eval_items)})")

# --- (1) plain classifier report ----------------------------------------------
head.eval()
ys, ps, probs = [], [], []
with torch.no_grad():
    for i0 in range(0, len(test_eval_items), BATCH):
        chunk = test_eval_items[i0:i0 + BATCH]
        for k in range(TEST_CROPS_PER_FILE):
            wavs = torch.stack([load_crop(p, deterministic_k=(k, TEST_CROPS_PER_FILE))
                                for p, _ in chunk]).to(DEVICE)
            pr = head(encode_pooled(backbone, wavs)).softmax(-1)
            probs += pr.cpu().tolist()
            ps += pr.argmax(-1).cpu().tolist()
            ys += [c for _, c in chunk]
print_report(ys, ps, f"TEST ({TEST_CROPS_PER_FILE} crops/file, \u2264{TEST_PER_CLASS}/class)")

# --- (2) routing view: pick τ on the *validation* set, report on test ---------
# B8 primary F1 per (true class, routed model): what a snippet of this class
# scores when sent to that model.
F1_TABLE = {  # class -> {"v2mix": F1, "base": F1}
    "electronic":       {"v2mix": 0.55, "base": 0.19},  # mean of corpus/nesmdb multi_f
    "classic":          {"v2mix": 0.868, "base": 0.949},
    "acoustic_band":    {"v2mix": 0.700, "base": 0.831},
    "electronic_drums": {"v2mix": 0.901, "base": 0.923},
}
BASE_CLASSES = {CLASSES.index("classic"), CLASSES.index("acoustic_band")}

def routed_f1(y_true, prob_rows, tau):
    tot = 0.0
    for t_, pr in zip(y_true, prob_rows):
        p_base = sum(pr[c] for c in BASE_CLASSES)
        model = "base" if p_base > tau else "v2mix"
        tot += F1_TABLE[CLASSES[t_]][model]
    return tot / len(y_true)

# τ sweep on a balanced validation subset (same IO reasoning as above)
val_sweep_items = balanced_subset(val_items, 137, seed=SEED + 1)
vys, vprobs = [], []
with torch.no_grad():
    for i0 in range(0, len(val_sweep_items), BATCH):
        chunk = val_sweep_items[i0:i0 + BATCH]
        wavs = torch.stack([load_crop(p, deterministic_k=(0, 1)) for p, _ in chunk]).to(DEVICE)
        vprobs += head(encode_pooled(backbone, wavs)).softmax(-1).cpu().tolist()
        vys += [c for _, c in chunk]

taus = np.linspace(0.3, 0.95, 27)
scores = [routed_f1(vys, vprobs, t) for t in taus]
TAU = float(taus[int(np.argmax(scores))])
perfect = np.mean([F1_TABLE[CLASSES[t_]][ "base" if t_ in BASE_CLASSES else "v2mix"] for t_ in ys])
always_v2 = routed_f1(ys, probs, 2.0)   # tau>1: never route to base
routed = routed_f1(ys, probs, TAU)
print(f"\nτ* = {TAU:.2f} (picked on validation)")
print(f"routed F1 on test:   {routed:.3f}")
print(f"  perfect router:    {perfect:.3f}")
print(f"  always-v2 (no clf):{always_v2:.3f}")
plt.plot(taus, scores); plt.axvline(TAU, ls="--", c="gray")
plt.xlabel("τ (route to base if P(base classes) > τ)"); plt.ylabel("routed F1 (val)")
plt.grid(alpha=0.3); plt.show()


## 8. Export — everything the GPU worker needs

`head_best.pt` (weights) + `classifier_meta.json` (classes, routing map, τ,
crop format, backbone identity). The worker-side integration loads the same
backbone it already has, attaches this head, and routes `model_version` by
`CLASS_TO_MODEL` — a separate, small PR.


In [ ]:
ts = time.strftime("%Y%m%d-%H%M%S")
export_dir = OUT_DIR / f"genre_head_{ts}"
export_dir.mkdir(parents=True, exist_ok=True)

torch.save(best["state"], export_dir / "head_best.pt")
meta = {
    "classes": CLASSES,
    "class_to_model": CLASS_TO_MODEL,
    "tau_route_to_base": TAU,
    "crop": {"seconds": CROP_SEC, "sample_rate": SR},
    "backbone": {"exp_id": RELEASED_EXP, "project": "2024", "arch": ARCH,
                 "pooling": "mean over segments and latent positions"},
    "val_macro_f1": best["f1"], "best_epoch": best["epoch"],
    "seed": SEED, "created": ts,
}
(export_dir / "classifier_meta.json").write_text(json.dumps(meta, indent=2))

# validation: reload from the exported files and re-score one test batch
head2 = GenreHead().to(DEVICE)
with torch.no_grad():   # materialize lazy layers before loading weights
    head2(encode_pooled(backbone, torch.zeros(1, CROP_SAMPLES, device=DEVICE)))
head2.load_state_dict(torch.load(export_dir / "head_best.pt"))
head2.eval()
chunk = test_items[:8]
wavs = torch.stack([load_crop(p, deterministic_k=(0, 1)) for p, _ in chunk]).to(DEVICE)
with torch.no_grad():
    agree = (head2(encode_pooled(backbone, wavs)).argmax(-1)
             == head(encode_pooled(backbone, wavs)).argmax(-1)).all().item()
print("export dir:", export_dir)
print("reload agreement on a test batch:", bool(agree))
